# Tennis match data preparation

## Goal

Prepare the Tennis-Data match history for a binary classifier that predicts whether `player_1` wins. The final objects (`X_train_ready`, `X_validation_ready`, `X_test_ready` and their targets) can be passed directly to a scikit-learn estimator.

## Setup

### Key assumptions

- Only information available before a match starts is used. Scores, sets, comments, and the original winner/loser ordering are excluded.
- Only completed matches are retained; retirements, walkovers, and awarded matches are not comparable sporting outcomes.
- Player sides are randomized with a fixed seed. This prevents the source convention (winner always stored first) from leaking the label.
- Splits are chronological: 2020–2024 train, 2025 validation, and 2026 test.
- Bookmaker odds are disabled by default. Enable them to build a bookmaker-informed benchmark rather than a rankings-only baseline.

In [10]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import assert_all_finite

RANDOM_SEED = 42
INCLUDE_BOOKMAKER_ODDS = False
TRAIN_END = datetime(2024, 12, 31)
VALIDATION_END = datetime(2025, 12, 31)

candidate_paths = [
    Path("../data/interim/tennis_matches_2020_2026.data"),
    Path("data/interim/tennis_matches_2020_2026.data"),
]
DATA_FILE = next((path.resolve() for path in candidate_paths if path.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError("Run this notebook from the project root or docs directory.")

print(f"Input: {DATA_FILE}")

Input: /home/albertocampagnolo/Desktop/tennis_prediction/data/interim/tennis_matches_2020_2026.data


## Data

### 1. Load and validate the source

In [11]:
if DATA_FILE is None:
    raise FileNotFoundError("Run this notebook from the project root or docs directory.")

raw_matches = pd.read_csv(DATA_FILE, low_memory=False)

required_columns = {
    "Date", "Tournament", "Series", "Court", "Surface", "Round",
    "Best of", "Winner", "Loser", "WRank", "LRank", "WPts",
    "LPts", "Comment", "AvgW", "AvgL",
}
missing_columns = required_columns.difference(raw_matches.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

raw_matches["Date"] = pd.to_datetime(raw_matches["Date"], errors="coerce")
print(f"Rows: {len(raw_matches):,}")
print(f"Date range: {raw_matches['Date'].min().date()} to {raw_matches['Date'].max().date()}")
raw_matches.head(3)

Rows: 16,031
Date range: 2020-01-06 to 2026-07-12


,source_year,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,...,B365W,B365L,PSW,PSL,MaxW,MaxL,AvgW,AvgL,BFEW,BFEL
0,2020,1,Doha,Qatar Exxon Mobil Open,2020-01-06,ATP250,Outdoor,Hard,1st Round,3.0,...,1.83,1.83,1.97,1.92,2.00,2.07,1.87,1.92,NaN,NaN
1,2020,1,Doha,Qatar Exxon Mobil Open,2020-01-06,ATP250,Outdoor,Hard,1st Round,3.0,...,2.0,1.72,2.21,1.74,2.25,1.80,2.11,1.72,NaN,NaN
2,2020,1,Doha,Qatar Exxon Mobil Open,2020-01-06,ATP250,Outdoor,Hard,1st Round,3.0,...,1.5,2.50,1.54,2.62,1.57,2.65,1.53,2.47,NaN,NaN


### 2. Keep valid, completed matches

The identity key removes accidental duplicate match records while preserving distinct meetings between the same players. Missing ranks and points are kept for later imputation.

In [12]:
match_key = ["Date", "Tournament", "Winner", "Loser"]
valid_matches = (
    raw_matches.loc[raw_matches["Comment"].eq("Completed")].copy()
    .dropna(subset=["Date", "Winner", "Loser"])
    .drop_duplicates(subset=match_key, keep="first")
    .sort_values(["Date", "Tournament", "Winner", "Loser"], kind="stable")
    .reset_index(drop=True)
)

numeric_source_columns = ["Best of", "WRank", "LRank", "WPts", "LPts", "AvgW", "AvgL"]
valid_matches[numeric_source_columns] = valid_matches[numeric_source_columns].apply(
    pd.to_numeric, errors="coerce"
)

print(f"Retained {len(valid_matches):,} of {len(raw_matches):,} rows ({len(valid_matches) / len(raw_matches):.1%}).")
valid_matches["Comment"].value_counts()

Retained 15,476 of 16,031 rows (96.5%).


Comment
Completed    15476
Name: count, dtype: int64

### 3. Remove winner-position leakage

Each source row stores winner attributes in `W...` columns and loser attributes in `L...` columns. A seeded random orientation assigns either player to `player_1`; `player_1_wins` is then the binary target.

In [13]:
rng = np.random.default_rng(RANDOM_SEED)
winner_is_player_1 = rng.random(len(valid_matches)) < 0.5

def orient(winner_values: pd.Series, loser_values: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    """Return values aligned to randomized player 1 and player 2 sides."""
    return (
        np.where(winner_is_player_1, winner_values, loser_values),
        np.where(winner_is_player_1, loser_values, winner_values),
    )

modeling_data = valid_matches[["Date", "Tournament", "Series", "Court", "Surface", "Round", "Best of"]].copy()
modeling_data.columns = ["match_date", "tournament", "series", "court", "surface", "round", "best_of"]
modeling_data["player_1"], modeling_data["player_2"] = orient(valid_matches["Winner"], valid_matches["Loser"])
modeling_data["player_1_rank"], modeling_data["player_2_rank"] = orient(valid_matches["WRank"], valid_matches["LRank"])
modeling_data["player_1_points"], modeling_data["player_2_points"] = orient(valid_matches["WPts"], valid_matches["LPts"])
modeling_data["avg_odds_1"], modeling_data["avg_odds_2"] = orient(valid_matches["AvgW"], valid_matches["AvgL"])
modeling_data["player_1_wins"] = winner_is_player_1.astype("int8")

modeling_data["rank_difference"] = modeling_data["player_1_rank"] - modeling_data["player_2_rank"]
modeling_data["points_difference"] = modeling_data["player_1_points"] - modeling_data["player_2_points"]
modeling_data["month"] = modeling_data["match_date"].dt.month.astype("int8")

modeling_data.head(3)

,match_date,tournament,series,court,surface,round,best_of,player_1,player_2,player_1_rank,player_2_rank,player_1_points,player_2_points,avg_odds_1,avg_odds_2,player_1_wins,rank_difference,points_difference,month
0,2020-01-06,Qatar Exxon Mobil Open,ATP250,Outdoor,Hard,1st Round,3.0,Ymer M.,Bedene A.,76.0,58.0,681.0,905.0,1.92,1.87,0,18.0,-224.0,1
1,2020-01-06,Qatar Exxon Mobil Open,ATP250,Outdoor,Hard,1st Round,3.0,Bublik A.,Mannarino A.,55.0,43.0,919.0,1111.0,2.11,1.72,1,12.0,-192.0,1
2,2020-01-06,Qatar Exxon Mobil Open,ATP250,Outdoor,Hard,1st Round,3.0,Barrere G.,Chardy J.,83.0,54.0,636.0,920.0,2.47,1.53,0,29.0,-284.0,1


### 4. Select pre-match features

Player names and tournament names are retained in `modeling_data` for inspection but are not baseline features. Excluding them avoids a large, sparse identity lookup and makes predictions possible for unseen players and events.

In [14]:
numeric_features = [
    "best_of", "player_1_rank", "player_2_rank",
    "player_1_points", "player_2_points",
    "rank_difference", "points_difference", "month",
]
if INCLUDE_BOOKMAKER_ODDS:
    numeric_features += ["avg_odds_1", "avg_odds_2"]

categorical_features = ["series", "court", "surface", "round"]
feature_columns = numeric_features + categorical_features
target_column = "player_1_wins"

modeling_data[feature_columns + [target_column]].isna().sum().to_frame("missing_values")

,missing_values
best_of,15
player_1_rank,8
player_2_rank,23
player_1_points,8
player_2_points,23
rank_difference,31
points_difference,31
month,0
series,0
court,0


### 5. Create chronological train, validation, and test sets

Random splitting would let the model learn from later matches when evaluated on earlier ones. These date boundaries simulate training a model and deploying it into future seasons.

In [15]:
train_mask = modeling_data["match_date"].le(TRAIN_END)
validation_mask = modeling_data["match_date"].gt(TRAIN_END) & modeling_data["match_date"].le(VALIDATION_END)
test_mask = modeling_data["match_date"].gt(VALIDATION_END)

def split_xy(mask: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    return (
        modeling_data.loc[mask, feature_columns].reset_index(drop=True),
        modeling_data.loc[mask, target_column].reset_index(drop=True),
    )

X_train, y_train = split_xy(train_mask)
X_validation, y_validation = split_xy(validation_mask)
X_test, y_test = split_xy(test_mask)

split_summary = pd.DataFrame({
    "rows": [len(X_train), len(X_validation), len(X_test)],
    "player_1_win_rate": [y_train.mean(), y_validation.mean(), y_test.mean()],
}, index=["train_2020_2024", "validation_2025", "test_2026"])
split_summary

,rows,player_1_win_rate
train_2020_2024,11442,0.500437
validation_2025,2490,0.508032
test_2026,1544,0.476684


## Results

### 6. Fit preprocessing on training data only

Numeric missing values are median-imputed and standardized. Categorical missing values are imputed and one-hot encoded. Fitting only on the training set prevents validation/test distribution information from leaking into preprocessing.

In [16]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

X_train_ready = preprocessor.fit_transform(X_train)
X_validation_ready = preprocessor.transform(X_validation)
X_test_ready = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

pd.DataFrame({
    "rows": [X_train_ready.shape[0], X_validation_ready.shape[0], X_test_ready.shape[0]],
    "transformed_features": [X_train_ready.shape[1]] * 3,
}, index=["train", "validation", "test"])

,rows,transformed_features
train,11442,26
validation,2490,26
test,1544,26


## Checks

These assertions catch target leakage, overlap between time periods, unbalanced orientation, and preprocessing failures before model training.

In [17]:
for forbidden in ["Winner", "Loser", "Wsets", "Lsets", "Comment"]:
    assert forbidden not in feature_columns
assert modeling_data.loc[train_mask, "match_date"].max() < modeling_data.loc[validation_mask, "match_date"].min()
assert modeling_data.loc[validation_mask, "match_date"].max() < modeling_data.loc[test_mask, "match_date"].min()
assert set(y_train.unique()) == {0, 1}
assert 0.45 <= y_train.mean() <= 0.55
assert X_train_ready.shape[0] == len(y_train)
assert X_validation_ready.shape[0] == len(y_validation)
assert X_test_ready.shape[0] == len(y_test)
assert X_train_ready.shape[1] == X_validation_ready.shape[1] == X_test_ready.shape[1]
assert_all_finite(X_train_ready)

print("All preparation checks passed.")
print(f"Training matrix: {X_train_ready.shape}; target: {y_train.shape}")

All preparation checks passed.
Training matrix: (11442, 26); target: (11442,)


## Next steps

The data is ready for training. Keep `preprocessor` together with the estimator in a scikit-learn `Pipeline` for deployment, tune models against the 2025 validation set, and use the 2026 test set only once for the final unbiased evaluation. A minimal estimator can now be trained with `model.fit(X_train_ready, y_train)`.